# The Agent Ladder: Detecting AI Agents That Pose as Your Customers

## 1. Setup

In [ ]:
"""Detecting AI agents in a Netomi virtual agent chat: the demo notebook.

All imports live in this first cell, because pylint reads the notebook as one module.
"""

# Section 1: start everything the demo needs and print the Netomi config.

import json

from agent_ladder import baseline
from agent_ladder.customer import check_ollama
from agent_ladder.results import print_level_result, print_results
from agent_ladder.runs import run_level
from agent_ladder.settings import load_settings
from agent_ladder.service import close_service_log
from agent_ladder.store import create_data_folder, delete_data_folder, save_human_sessions
from agent_ladder.tunnel import PAGE_PORT, SERVICE_PORT
from agent_ladder.tunnel import start_page_server, start_service, start_tunnel, stop_all

settings = load_settings()

check_ollama()
print()

data_folder = create_data_folder()
print("Data folder, where telemetry events, signals, verdicts and session labels")
print("are appended as JSONL files:")
print(f"  {data_folder}")
print()

service = start_service(settings["bearer_token"])
print(f"Detection service is running on http://localhost:{SERVICE_PORT}.")
print("  /telemetry receives events from the demo page.")
print("  /mcp answers the VA's check_session and report_signal calls.")
print()

page_server = start_page_server(settings["netomi_channel_ref_id"])
print("Demo page (the Netomi widget plus collector.js). Human testers open this,")
print("and the harness drives it for Levels 3-6:")
print(f"  http://localhost:{PAGE_PORT}/")
print()

public_url = start_tunnel(settings["ngrok_domain"])
print("Public URL through the ngrok tunnel, which is how Netomi reaches the detection service:")
print(f"  {public_url}")
print()

mcp_config = {
    "detector": {
        "url": f"{public_url}/mcp",
        "transport": "streamable_http",
        "headers": {
            "Authorization": f"Bearer {settings['bearer_token']}",
            "User-Agent": "netomi",
        },
    },
}
print("MCP server config to paste into Netomi. The URL and token come from .env,")
print("so this stays the same across runs:")
print(json.dumps(mcp_config, indent=2))

## 2. Configure Netomi and Record the Human Sessions

This section has no code. Do these steps by hand, in order, while Section 1's servers are running.

**Configure Netomi** (in Agentic Studio):

1. **MCP server:** add an MCP server and paste the config Section 1 printed (Streamable HTTP, the bearer token, `User-Agent: netomi`). Refresh its tool list and confirm `check_session` and `report_signal` appear, and are active in Staging.
2. **Instructions:** paste `assets/netomi/agent-instructions.md`, its Response Instruction and Adjust Response parts into the fields of the same names. The "Knowledge tool: Order Guide" part goes with Adjust Response; its heading must match the knowledge tool's name in Netomi.
3. **Knowledge source:** upload `assets/netomi/knowledge.pdf` and make it active in Staging; until it is, the VA finds no orders. It holds order 48213 and the delivery-day facts the VA answers from. Order records in a knowledge source are a demo shortcut: a real deployment would look orders up in a database through a tool. The demo only needs the VA to give a consistent answer to one order question, so a knowledge source keeps it simple.
4. **Order playbook:** create a Text playbook (Build → AI Agent Tools → Playbooks → Add → Text) and follow the steps at the top of `assets/netomi/order-playbook.md`: paste each part into its field, insert `DETECT_SERVICES/CHECK_SESSION`, `DETECT_SERVICES/REPORT_SIGNAL` and the `DETECT_SESSION_ID` variable with the @ picker, click Validate, then Move to Staging.

## 3. The Human Baseline

In [ ]:
# Section 3: save and summarize the human baseline, and freeze its thresholds.

newly_saved = save_human_sessions()
human_sessions = baseline.load_human_sessions()
print(f"Human baseline: {len(human_sessions)} sessions in assets/humans/ "
      f"({len(newly_saved)} added just now; "
      f"the plan calls for {baseline.HUMAN_SESSIONS_EXPECTED}).")
print()

if not human_sessions:
    raise RuntimeError("No human sessions yet. Open http://localhost:8001/?label=human, "
                       "chat with the VA (see Section 2), then run this cell again.")

features_by_session = {}
for session_id, events in human_sessions.items():
    features = baseline.input_features(events)
    if features is None:
        print(f"Skipped session {session_id[:8]}: it has too little typing to measure.")
        continue
    features_by_session[session_id] = features

# Frozen once, from the first full baseline, and reused for every later demo.
input_stats_thresholds = baseline.load_thresholds()
if input_stats_thresholds is None:
    input_stats_thresholds = baseline.compute_thresholds(features_by_session)
    baseline.freeze_thresholds(input_stats_thresholds, len(features_by_session))
    print("Thresholds computed from these sessions and frozen just now.")
else:
    print("Thresholds frozen earlier and reused, so nothing is tuned between demos.")
baseline.print_baseline_table(features_by_session, input_stats_thresholds)

## 4. Level 1: Declared

In [ ]:
# Section 4: Level 1, an agent that says it's an AI when asked.

level_1_sessions = run_level("1-declared")
print()
print_level_result(1)

## 5. Level 2: Direct API

In [ ]:
# Section 5: Level 2, an agent posting straight to the Chat API.

level_2_sessions = run_level("2-direct-api")
print()
print_level_result(2)

## 6. Level 3: Naive Browser

In [ ]:
# Section 6: Level 3, Playwright with a default launch and fill().

level_3_sessions = run_level("3-naive-browser")
print()
print_level_result(3)

## 7. Level 4: Masked Automation

In [ ]:
# Section 7: Level 4, a stealth launch that hides the automation flags.

level_4_sessions = run_level("4-masked")
print()
print_level_result(4)

## 8. Level 5: Human-Like Input

In [ ]:
# Section 8: Level 5, key-by-key typing with human-like timing.

level_5_sessions = run_level("5-human-input")
print()
print_level_result(5)

## 9. Level 6: Human-Like Language

In [ ]:
# Section 9: Level 6, human-like typing and casual, short messages.

level_6_sessions = run_level("6-human-language")
print()
print_level_result(6, show_va_catches=True)

## 10. Results

In [ ]:
# Section 10: compare every session's true label with the detector's verdict.

print_results()

## 11. Cleanup

In [ ]:
# Section 11: stop everything Section 1 started and delete this demo's records.

stop_all(service, page_server)
print("Stopped the ngrok tunnel, the demo page server and the detection service.")
print(f"Ports {SERVICE_PORT} and {PAGE_PORT} are free, so Section 1 can be run again.")
print()

close_service_log()
delete_data_folder()
print(f"Deleted {data_folder}: this demo's events, signals, verdicts, runs and service.log.")
print("The next demo starts with empty records. The human baseline in assets/humans/")
print("(sessions, their verdicts and the frozen thresholds) is kept, so the testers")
print("don't need to be recorded again.")